# Phishing Website Detection — Experimental Analysis

**Data source:** [UCI Phishing Websites, dataset 327](https://archive.ics.uci.edu/dataset/327/phishing+websites) — 11,055 real labeled website observations (30 numeric features).

The JSONL rows here are **new local measurements** of model inference on UCI observations, not pre-existing HTTP server access logs. Training and warm-up occur before benchmark statistics.

**Reproduction:** Run `python -m src.download_uci`, then `python -m src.collect_inference --data data/external/uci327_phishing_websites.csv --runs 120 --out data/raw/test_run.jsonl` from repository root. Then use *Restart & Run All*.

Do not submit previous 10-row smoke-test output or constant-latency dump as real measurements. Notebook outputs are intentionally cleared until actual UCI logs exist.


# Reproducible ML Inference Benchmark

**Goal:** process raw JSONL measurements, exclude cold-start warm-up runs, calculate error rates and seeded bootstrap uncertainty, and export publication-style figures. **Dataset:** UCI 327 (11,055 real labeled website records). Results are generated only after running the real measurement collector on Linux.

## 1. Environment and reproducibility
Run **Kernel → Restart & Run All**. All paths resolve from the project root. Randomness in Python and NumPy is fixed to **42**; bootstrap sampling uses a separately seeded local generator inside `calculate_statistics`. CSV and graphics are written with deterministic filenames.

In [1]:
from pathlib import Path
import sys, random
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.process_data import read_telemetry, load_and_clean_data, calculate_statistics
from src.visualize import generate_plots
SEED=42
random.seed(SEED)
np.random.seed(SEED)
print("Project:", ROOT, "| Seed:", SEED)

/home/oai/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-4yfyzdcn because there was an issue with the default path (/home/oai/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Project: /tmp/audit_j00vl47v/phishing-ml-benchmark | Seed: 42


## 2. Data ingestion and schema validation
Each JSONL record must supply `phase`, `model_variant`, `batch_size`, `latency_ms`, and `status_code`. `batch_size` is **not** requests-per-second. The locally measured `test_run.jsonl` is excluded from Git; it must be generated with the UCI dataset before executing this notebook.

**GitHub clone behavior:** If the full local `test_run.jsonl` is missing, the notebook loads the committed 130-call `test_run_sample.jsonl` as a smoke-test preview. Preview outputs are stored separately and never overwrite the full-run summary or publication figures. For the full experiment run `bash run_dataset.sh` first.

In [2]:
FULL = ROOT / "data/raw/test_run.jsonl"
SAMPLE = ROOT / "data/raw/test_run_sample.jsonl"
IS_SAMPLE = not FULL.exists()
RAW = SAMPLE if IS_SAMPLE else FULL
if not RAW.exists():
    raise FileNotFoundError("No telemetry available: run bash run_dataset.sh to collect real measurements.")
print("Input:", RAW.relative_to(ROOT))
if IS_SAMPLE:
    print("SAMPLE PREVIEW ONLY — these 130 recorded calls do not reproduce the published full-run statistics.")
    print("Run bash run_dataset.sh to create the complete raw log, then Restart & Run All.")
raw = read_telemetry(RAW)
print("Total observations:", len(raw))
print("Phase counts:")
display(raw.groupby("phase").size().rename("records").to_frame())
print("Model / batch counts:")
display(raw.groupby(["model_variant","batch_size","phase"]).size().unstack(fill_value=0))

Input: data/raw/test_run_sample.jsonl
SAMPLE PREVIEW ONLY — these 130 recorded calls do not reproduce the published full-run statistics.
Run bash run_dataset.sh to create the complete raw log, then Restart & Run All.
Total observations: 130
Phase counts:


,records
phase,
benchmark,100
warmup,30


Model / batch counts:


phase                          benchmark  warmup
model_variant      batch_size                   
LogisticRegression 1                  10       3
                   2                  10       3
                   4                  10       3
                   6                  10       3
                   9                  10       3
RandomForest       1                  10       3
                   2                  10       3
                   4                  10       3
                   6                  10       3
                   9                  10       3

## 3. Warm-up filtering and failure handling
Warm-up measurements have cold-cache/start-up bias and must be excluded from benchmark statistics. **Failures must not disappear:** they remain in each group’s error-rate denominator; only valid successful calls contribute to latency percentiles.

In [3]:
benchmark = load_and_clean_data(RAW)
n_warmup=int((raw.phase=="warmup").sum())
assert n_warmup > 0, "Fixture must exercise warm-up removal"
assert len(benchmark)+n_warmup == len(raw)
assert benchmark.phase.eq("benchmark").all()
print(f"Excluded warm-up: {n_warmup}; retained benchmark rows: {len(benchmark)}")
print(f"Benchmark failures: {(benchmark.status_code!=200).sum()}")

Excluded warm-up: 30; retained benchmark rows: 100
Benchmark failures: 0


## 4. Descriptive statistics + 95% bootstrap CI
For each model and batch size: calculate sample count, failed calls, error rate (%), mean ± SD, median, p25/p75, p95 and p99. Bootstrap **2,000** re-sampled medians with replacement. Use the 2.5th and 97.5th percentiles as a percentile 95% CI, with seed **42**. Short sequential timings may be correlated, so this is an illustrative uncertainty interval.

In [4]:
summary=calculate_statistics(benchmark,seed=SEED,n_boot=2000)
assert (summary.n_total==summary.n_success+summary.n_error).all()
assert (summary.median_ci_lower_ms<=summary.median_ms).all()
assert (summary.median_ms<=summary.median_ci_upper_ms).all()
display(summary.round(4))
CSV=ROOT/("data/processed/sample_metrics_summary.csv" if IS_SAMPLE else "data/processed/metrics_summary.csv")
CSV.parent.mkdir(parents=True,exist_ok=True)
summary.to_csv(CSV,index=False)
print("Saved:",CSV, "| sample preview only" if IS_SAMPLE else "| complete experiment")

,model_variant,batch_size,n_total,n_success,n_error,error_rate_pct,mean_ms,sd_ms,median_ms,p25_ms,p75_ms,p95_ms,p99_ms,median_ci_lower_ms,median_ci_upper_ms
0,LogisticRegression,1,10,10,0,0.0,2.1997,0.0199,2.1993,2.1878,2.2061,2.2305,2.2383,2.1862,2.2127
1,LogisticRegression,2,10,10,0,0.0,1.7563,0.0589,1.7455,1.7290,1.7575,1.8516,1.8959,1.7251,1.7696
2,LogisticRegression,4,10,10,0,0.0,1.7312,0.0564,1.7104,1.6992,1.7352,1.8312,1.8625,1.6973,1.7479
3,LogisticRegression,6,10,10,0,0.0,1.6765,0.0877,1.6806,1.5972,1.7224,1.8007,1.8367,1.5820,1.7238
4,LogisticRegression,9,10,10,0,0.0,1.6677,0.0756,1.6874,1.5922,1.7300,1.7589,1.7669,1.5893,1.7366
5,RandomForest,1,10,10,0,0.0,13.9114,0.0923,13.9158,13.8527,13.9775,14.0359,14.0520,13.8294,13.9886
6,RandomForest,2,10,10,0,0.0,15.2137,1.3426,14.9387,13.9695,16.3959,16.9037,17.0568,13.9263,16.4301
7,RandomForest,4,10,10,0,0.0,14.0350,0.2220,13.9713,13.8918,14.1124,14.3927,14.5485,13.8885,14.1486
8,RandomForest,6,10,10,0,0.0,14.6973,0.4674,14.5352,14.3942,15.1300,15.3656,15.4038,14.3625,15.3041
9,RandomForest,9,10,10,0,0.0,14.8003,0.4736,14.6247,14.4383,15.0079,15.6015,15.6653,14.4137,15.1641


Saved: /tmp/audit_j00vl47v/phishing-ml-benchmark/data/processed/sample_metrics_summary.csv | sample preview only


## 5. Scientific visualization
**Figure 1:** median latency (ms) vs batch size, with **95% bootstrap CI error bars**, distinct markers and line styles. **Figure 2:** boxplot of latency distributions for the two model variants (pooled across all batch sizes; warm-up excluded). Axis units are explicit. Both plots are saved as **vector PDF and 300 DPI PNG**.

In [5]:
FIG=ROOT/("reports/figures/sample_preview" if IS_SAMPLE else "reports/figures")
generate_plots(benchmark,FIG,seed=SEED)
files=sorted(FIG.glob("fig*"))
for f in files:print(f.relative_to(ROOT),f.stat().st_size,"bytes")
assert all((FIG/f"{name}.{ext}").is_file() for name in ["fig1_latency_vs_load","fig2_distribution_boxplot"] for ext in ["pdf","png"])

reports/figures/sample_preview/fig1_latency_vs_load.pdf 18279 bytes
reports/figures/sample_preview/fig1_latency_vs_load.png 134313 bytes
reports/figures/sample_preview/fig2_distribution_boxplot.pdf 16190 bytes
reports/figures/sample_preview/fig2_distribution_boxplot.png 118585 bytes


## 6. Results & Discussion (limitations)
This notebook analyzes **actual local inference measurements on UCI 327**, once generated. It does not establish which model is superior in phishing classification: this benchmark measures the predict latency only, not held-out F1 or recall. Batch size is not production RPS; timings depend on Linux hardware, background noise, and CPU scheduling. Report model quality separately and avoid claiming production traffic traces.parately from inference timing. See `docs/domain_model.md` for mathematical definitions.

A sample-only notebook run demonstrates reproducibility of code, **not** reproducibility of the full experimental estimates. To reproduce the published estimates, collect the entire log on the same target environment and rerun all cells.